In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Step 1: Fault Labels

In [3]:
canopy_2015 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2015_Labeled.csv")
canopy_2016 = pd.read_csv("/content/drive/MyDrive/NIST_Preprocessed/canopy_2016_Labeled.csv")

In [4]:
# Convert the TIMESTAMP column to pandas datetime object
canopy_2015['TIMESTAMP'] = pd.to_datetime(canopy_2015['TIMESTAMP'])
canopy_2016['TIMESTAMP'] = pd.to_datetime(canopy_2016['TIMESTAMP'])

# Set TIMESTAMP as index
canopy_2015.set_index('TIMESTAMP', inplace=True)
canopy_2016.set_index('TIMESTAMP', inplace=True)

In [5]:
# Fault Indicator Columns
# Rather than merging label_string/label_inverter into a single priority-ordered
# column (which would hide genuine co-occurrence between fault types), we keep
# both label sets fully separate and add lightweight boolean indicators on top.
# This supports both a coarse "fault vs no-fault" detection task and full
# subsystem-level classification, without discarding information.

NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2015['has_string_fault']   = ~canopy_2015['label_string'].isin(NON_FAULT_VALUES)
canopy_2015['has_inverter_fault'] = ~canopy_2015['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2015['has_any_fault']      = canopy_2015['has_string_fault'] | canopy_2015['has_inverter_fault']
canopy_2015['fault_cooccurrence'] = canopy_2015['has_string_fault'] & canopy_2015['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2015) ===\n")
print(f"has_string_fault   : {canopy_2015['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2015['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2015['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2015['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2015.loc[canopy_2015['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2015) ===

has_string_fault   : 11146
has_inverter_fault : 10380
has_any_fault      : 20801
fault_cooccurrence : 725

As % of evaluable rows (Op_Mask=1): 10.92%


In [6]:
NON_FAULT_VALUES = ['NORMAL', 'NOT_EVALUABLE', 'MISSING_DATA', 'INSUFFICIENT_DATA']

canopy_2016['has_string_fault']   = ~canopy_2016['label_string'].isin(NON_FAULT_VALUES)
canopy_2016['has_inverter_fault'] = ~canopy_2016['label_inverter'].isin(NON_FAULT_VALUES)
canopy_2016['has_any_fault']      = canopy_2016['has_string_fault'] | canopy_2016['has_inverter_fault']
canopy_2016['fault_cooccurrence'] = canopy_2016['has_string_fault'] & canopy_2016['has_inverter_fault']

# Summary
print("=== Fault Indicator Summary (2016) ===\n")
print(f"has_string_fault   : {canopy_2016['has_string_fault'].sum()}")
print(f"has_inverter_fault : {canopy_2016['has_inverter_fault'].sum()}")
print(f"has_any_fault      : {canopy_2016['has_any_fault'].sum()}")
print(f"fault_cooccurrence : {canopy_2016['fault_cooccurrence'].sum()}")

print(f"\nAs % of evaluable rows (Op_Mask=1): "
      f"{canopy_2016.loc[canopy_2016['Op_Mask']==1, 'has_any_fault'].mean() * 100:.2f}%")

=== Fault Indicator Summary (2016) ===

has_string_fault   : 7610
has_inverter_fault : 11428
has_any_fault      : 18442
fault_cooccurrence : 596

As % of evaluable rows (Op_Mask=1): 9.54%


In [7]:
columns = ['Op_Mask', 'InvPAC_kW_Avg', 'has_any_fault']
canopy_2015[columns].to_csv('/content/drive/MyDrive/CS_LSTM/canopy_2015_normal_hour_analysis.csv', index=True)

#Step 2: Build Healthy Mask

In [19]:
def create_healthy_mask(df):

    return (
        (df["Op_Mask"] == 1) &
        (df["InvOpStatus_Avg"] > 0) &
        (~df["has_any_fault"])
    )


healthy_mask_2015 = create_healthy_mask(canopy_2015)
healthy_mask_2016 = create_healthy_mask(canopy_2016)

In [20]:
print(f"Healthy Operating Periods (2015): {healthy_mask_2015.sum()}")
print(f"Healthy Operating Periods (2016): {healthy_mask_2016.sum()}")

Healthy Operating Periods (2015): 169665
Healthy Operating Periods (2016): 174843


#Step 3: Segment Analysis

In [21]:
def analyze_segments(mask, label, seq_len=60):

    groups = (mask != mask.shift()).cumsum()

    segment_lengths = (
        mask.groupby(groups)
            .sum()
    )

    segment_lengths = segment_lengths[segment_lengths > 0]

    usable = (segment_lengths - seq_len + 1).clip(lower=0)

    print(f"\n========== {label} ==========")
    print(f"Healthy timestamps            : {segment_lengths.sum():,}")
    print(f"Continuous healthy segments   : {len(segment_lengths):,}")
    print(f"Longest segment               : {segment_lengths.max():,}")
    print(f"Average segment               : {segment_lengths.mean():.1f}")
    print(f"Segments < {seq_len} samples  : {(usable==0).sum():,}")
    print(f"Usable windows (L={seq_len})  : {usable.sum():,}")

    return segment_lengths

In [22]:
segments_2015 = analyze_segments(healthy_mask_2015,"2015")
segments_2016 = analyze_segments(healthy_mask_2016,"2016")


========== 2015 ==========
Healthy timestamps            : 169,665
Continuous healthy segments   : 3,552
Longest segment               : 678
Average segment               : 47.8
Segments < 60 samples  : 2,956
Usable windows (L=60)  : 92,539

========== 2016 ==========
Healthy timestamps            : 174,843
Continuous healthy segments   : 4,107
Longest segment               : 662
Average segment               : 42.6
Segments < 60 samples  : 3,527
Usable windows (L=60)  : 92,366


In [23]:
# Feature Selection & Contextual Encoding

PRIMARY_FEATURES = [
    'POA_fused',
    'AmbTemp_C_Avg',
    'module_temp',
    'InvVDCin_Avg',
    'InvIDCin_Avg',
    'InvPDC_kW_Avg',
    'I_string_mean',
    'InvPAC_kW_Avg_lag1',
]

CONTEXTUAL_FEATURES = [
    'sin_doy',
    'cos_doy',
    'sin_hour',
    'cos_hour',
]

TARGET = 'InvPAC_kW_Avg'


def add_features(df):
    df = df.copy()

    # Autoregressive feature
    df['InvPAC_kW_Avg_lag1'] = df['InvPAC_kW_Avg'].shift(1)

    # Contextual features
    if 'TIMESTAMP' in df.columns:
      df = df.set_index(pd.to_datetime(df['TIMESTAMP']))

    doy  = df.index.day_of_year.values
    hour = df.index.hour.values + df.index.minute.values / 60.0

    df['sin_doy']  = np.sin(2 * np.pi * doy  / 365.0)
    df['cos_doy']  = np.cos(2 * np.pi * doy  / 365.0)
    df['sin_hour'] = np.sin(2 * np.pi * hour / 24.0)
    df['cos_hour'] = np.cos(2 * np.pi * hour / 24.0)

    return df

In [24]:
canopy_2015 = add_features(canopy_2015)
canopy_2016 = add_features(canopy_2016)

#Step 4: Extract Continuous Healthy Segments

In [26]:
def extract_segments(df, healthy_mask):

    groups = (healthy_mask != healthy_mask.shift()).cumsum()

    segments = []

    for _, segment in df.groupby(groups):

        if not healthy_mask.loc[segment.index[0]]:
            continue

        segments.append(segment.copy())

    return segments

In [27]:
segments2015 = extract_segments(
    canopy_2015,
    healthy_mask_2015
)

segments2016 = extract_segments(
    canopy_2016,
    healthy_mask_2016
)

In [28]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

2015: 3552 segments
2016: 4107 segments


#Stage 5: Remove first fow of every segment

In [29]:
def clean_segments(segments):

    cleaned = []

    removed = 0

    for seg in segments:

        if len(seg) <= 1:
            continue

        cleaned.append(seg.iloc[1:].copy())
        removed += 1

    print(f"Removed first row from {removed:,} segments.")

    return cleaned

In [30]:
segments2015 = clean_segments(segments2015)
segments2016 = clean_segments(segments2016)

Removed first row from 3,117 segments.
Removed first row from 3,552 segments.


#Stage 6: Remove short segments

In [31]:
SEQ_LEN = 60

segments2015 = [
    s for s in segments2015
    if len(s) >= SEQ_LEN
]

segments2016 = [
    s for s in segments2016
    if len(s) >= SEQ_LEN
]

In [32]:
print(f"2015: {len(segments2015)} segments")
print(f"2016: {len(segments2016)} segments")

2015: 586 segments
2016: 570 segments


#Results

In [33]:
print("=" * 60)
print("Healthy Segment Summary")
print("=" * 60)

print(f"2015:")
print(f"  Total healthy segments : {len(segments2015):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2015):,}")

print()

print(f"2016:")
print(f"  Total healthy segments : {len(segments2016):,}")
print(f"  Total healthy samples  : {sum(len(s) for s in segments2016):,}")

Healthy Segment Summary
2015:
  Total healthy segments : 586
  Total healthy samples  : 126,517

2016:
  Total healthy segments : 570
  Total healthy samples  : 125,416


In [36]:
import pandas as pd

lengths2015 = pd.Series([len(s) for s in segments2015])
lengths2016 = pd.Series([len(s) for s in segments2016])

print("========== 2015 ==========")
print(lengths2015.describe())

print("\n========== 2016 ==========")
print(lengths2016.describe())

========== 2015 ==========
count    586.000000
mean     215.899317
std      150.829723
min       60.000000
25%       84.250000
50%      164.500000
75%      322.000000
max      677.000000
dtype: float64

========== 2016 ==========
count    570.00000
mean     220.02807
std      148.13755
min       60.00000
25%       88.00000
50%      169.50000
75%      331.00000
max      661.00000
dtype: float64


In [37]:
lengths2015 = pd.Series([len(s) for s in segments2015])

print("Top 10 longest healthy segments (2015):")

display(
    lengths2015.sort_values(ascending=False)
               .head(10)
               .rename("Length")
               .to_frame()
)

Top 10 longest healthy segments (2015):


,Length
199,677
236,670
197,660
177,639
150,628
237,606
129,584
395,579
300,578
411,575


In [44]:
segment_id = 300

segment = segments2015[segment_id]

print(f"Segment {segment_id}")
print(f"Rows      : {len(segment)}")
print(f"Start     : {segment.index[0]}")
print(f"End       : {segment.index[-1]}")

display(segment)

Segment 300
Rows      : 578
Start     : 2015-07-07 07:10:00-05:00
End       : 2015-07-07 16:47:00-05:00


,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,...,inv_shutdown_status_flag,has_string_fault,has_inverter_fault,has_any_fault,fault_cooccurrence,InvPAC_kW_Avg_lag1,sin_doy,cos_doy,sin_hour,cos_hour
TIMESTAMP,,,,,,,,,,,,,,,,,,,,,
2015-07-07 07:10:00-05:00,105.034820,99.726780,104.92843,108.0,106.1,22.34,336.0,273.5,274.3,275.6,...,0,False,False,False,False,23.19,-0.094537,-0.995521,0.953717,-0.300706
2015-07-07 07:11:00-05:00,109.999886,104.898520,110.08793,112.8,111.2,22.34,336.3,273.5,274.1,275.3,...,0,False,False,False,False,24.22,-0.094537,-0.995521,0.952396,-0.304864
2015-07-07 07:12:00-05:00,114.964950,109.972690,115.35960,117.7,116.1,22.29,337.3,273.3,274.1,275.3,...,0,False,False,False,False,25.39,-0.094537,-0.995521,0.951057,-0.309017
2015-07-07 07:13:00-05:00,120.639330,115.534744,120.96774,122.0,121.3,22.33,337.5,273.3,274.0,275.2,...,0,False,False,False,False,26.56,-0.094537,-0.995521,0.949699,-0.313164
2015-07-07 07:14:00-05:00,127.259415,121.877450,127.47319,126.9,127.3,22.38,337.5,273.1,273.9,275.1,...,0,False,False,False,False,27.86,-0.094537,-0.995521,0.948324,-0.317305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-07-07 16:43:00-05:00,460.391780,388.075740,365.14648,435.6,331.1,31.56,330.2,271.5,272.5,274.6,...,0,False,False,False,False,86.30,-0.094537,-0.995521,-0.944089,-0.329691
2015-07-07 16:44:00-05:00,454.008150,382.220950,359.65048,429.2,326.0,31.64,331.1,271.5,272.6,274.8,...,0,False,False,False,False,85.70,-0.094537,-0.995521,-0.945519,-0.325568
2015-07-07 16:45:00-05:00,448.924870,377.537080,355.27612,424.5,321.9,31.53,329.9,271.8,272.6,274.8,...,0,False,False,False,False,84.40,-0.094537,-0.995521,-0.946930,-0.321439


#Windowing

In [46]:
SEQ_LEN = 60      # 60 minutes
HORIZON = 1       # Predict next timestep

print(f"Sequence length : {SEQ_LEN}")
print(f"Prediction horizon : {HORIZON}")

Sequence length : 60
Prediction horizon : 1


In [47]:
def create_windows_from_segment(
    segment,
    primary_features,
    contextual_features,
    target,
    seq_len=60,
    horizon=1
):
    """
    Generate CS-LSTM windows from one continuous healthy segment.

    Returns
    -------
    X_primary : (N, seq_len, n_primary)
    X_context : (N, seq_len, n_context)
    y         : (N,)
    """

    X_primary = []
    X_context = []
    y = []

    n = len(segment)

    for start in range(n - seq_len - horizon + 1):

        end = start + seq_len

        X_primary.append(
            segment.iloc[start:end][primary_features].values
        )

        X_context.append(
            segment.iloc[start:end][contextual_features].values
        )

        y.append(
            segment.iloc[end + horizon - 1][target]
        )

    return X_primary, X_context, y

In [48]:
import numpy as np

def create_training_dataset(
    segments,
    primary_features,
    contextual_features,
    target,
    seq_len=60,
    horizon=1
):

    Xp = []
    Xc = []
    y = []

    total_segments = len(segments)
    skipped = 0

    for segment in segments:

        # Remove the first row (lag belongs to previous segment)
        segment = segment.iloc[1:]

        if len(segment) < seq_len + horizon:
            skipped += 1
            continue

        xp, xc, yy = create_windows_from_segment(
            segment,
            primary_features,
            contextual_features,
            target,
            seq_len,
            horizon
        )

        Xp.extend(xp)
        Xc.extend(xc)
        y.extend(yy)

    Xp = np.asarray(Xp, dtype=np.float32)
    Xc = np.asarray(Xc, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    print("="*60)
    print("Sliding Window Summary")
    print("="*60)
    print(f"Healthy segments            : {total_segments:,}")
    print(f"Skipped (<{seq_len+1} rows) : {skipped:,}")
    print(f"Used segments               : {total_segments-skipped:,}")
    print(f"Generated windows           : {len(y):,}")
    print()

    print(f"Primary tensor   : {Xp.shape}")
    print(f"Context tensor   : {Xc.shape}")
    print(f"Target vector    : {y.shape}")

    return Xp, Xc, y

In [49]:
Xp2015, Xc2015, y2015 = create_training_dataset(
    segments2015,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 586
Skipped (<61 rows) : 19
Used segments               : 567
Generated windows           : 90,780

Primary tensor   : (90780, 60, 8)
Context tensor   : (90780, 60, 4)
Target vector    : (90780,)


In [50]:
Xp2016, Xc2016, y2016 = create_training_dataset(
    segments2016,
    PRIMARY_FEATURES,
    CONTEXTUAL_FEATURES,
    TARGET,
    seq_len=SEQ_LEN,
    horizon=HORIZON
)

Sliding Window Summary
Healthy segments            : 570
Skipped (<61 rows) : 17
Used segments               : 553
Generated windows           : 90,658

Primary tensor   : (90658, 60, 8)
Context tensor   : (90658, 60, 4)
Target vector    : (90658,)


In [51]:
Xp_train = np.concatenate([Xp2015, Xp2016], axis=0)
Xc_train = np.concatenate([Xc2015, Xc2016], axis=0)
y_train = np.concatenate([y2015, y2016], axis=0)

print("="*60)
print("Final Training Dataset")
print("="*60)

print(f"Primary tensor : {Xp_train.shape}")
print(f"Context tensor : {Xc_train.shape}")
print(f"Targets        : {y_train.shape}")

Final Training Dataset
Primary tensor : (181438, 60, 8)
Context tensor : (181438, 60, 4)
Targets        : (181438,)
